In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib
import warnings

In [6]:
warnings.filterwarnings('ignore')
path = r'C:\Users\anmol\Downloads\crop_yield.csv'
try:
    df = pd.read_csv(path)
except FileNotFoundError:
    print("Error: 'crop_yield_data.csv' not found. Please check the file path.")
    exit()

In [7]:
print("Initial data shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
print(df.head())
print("\nDataset information:")
df.info()

Initial data shape: (19689, 10)

First 5 rows of the dataset:
           Crop  Crop_Year       Season  State     Area  Production  \
0      Arecanut       1997  Whole Year   Assam  73814.0       56708   
1     Arhar/Tur       1997  Kharif       Assam   6637.0        4685   
2   Castor seed       1997  Kharif       Assam    796.0          22   
3      Coconut        1997  Whole Year   Assam  19656.0   126905000   
4  Cotton(lint)       1997  Kharif       Assam   1739.0         794   

   Annual_Rainfall  Fertilizer  Pesticide        Yield  
0           2051.4  7024878.38   22882.34     0.796087  
1           2051.4   631643.29    2057.47     0.710435  
2           2051.4    75755.32     246.76     0.238333  
3           2051.4  1870661.52    6093.36  5238.051739  
4           2051.4   165500.63     539.09     0.420909  

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 10 columns):
 #   Column           Non-Null Count  

Pre-Processing

In [8]:
df.drop(['Production'], axis=1, inplace=True)
print("\nMissing values before cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
Crop               0
Crop_Year          0
Season             0
State              0
Area               0
Annual_Rainfall    0
Fertilizer         0
Pesticide          0
Yield              0
dtype: int64


In [9]:
categorical_features = ['Crop', 'State', 'Season']
df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print("\nData shape after one-hot encoding:", df.shape)
print("\nFirst 5 rows of the preprocessed data:")
print(df.head())


Data shape after one-hot encoding: (19689, 94)

First 5 rows of the preprocessed data:
   Crop_Year     Area  Annual_Rainfall  Fertilizer  Pesticide        Yield  \
0       1997  73814.0           2051.4  7024878.38   22882.34     0.796087   
1       1997   6637.0           2051.4   631643.29    2057.47     0.710435   
2       1997    796.0           2051.4    75755.32     246.76     0.238333   
3       1997  19656.0           2051.4  1870661.52    6093.36  5238.051739   
4       1997   1739.0           2051.4   165500.63     539.09     0.420909   

   Crop_Arhar/Tur  Crop_Bajra  Crop_Banana  Crop_Barley  ...  State_Telangana  \
0           False       False        False        False  ...            False   
1            True       False        False        False  ...            False   
2           False       False        False        False  ...            False   
3           False       False        False        False  ...            False   
4           False       False        F

Model Training

In [11]:
X = df.drop('Yield', axis=1)
y = df['Yield']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining and Testing data shapes:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


Training and Testing data shapes:
X_train shape: (15751, 93)
X_test shape: (3938, 93)
y_train shape: (15751,)
y_test shape: (3938,)


In [12]:
print("\nTraining the RandomForestRegressor model...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model training complete.")


Training the RandomForestRegressor model...
Model training complete.


Model Evaluation

In [13]:
y_pred = model.predict(X_test)

# Calculate performance metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("\nModel Evaluation on Test Set:")
print(f"R-squared (R²): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")


Model Evaluation on Test Set:
R-squared (R²): 0.9805
Mean Absolute Error (MAE): 9.4215
Mean Squared Error (MSE): 15593.8761


Saving the Model

In [16]:
model_filename = 'random_forest_model.joblib'
columns_filename = 'model_features.joblib'

joblib.dump(model, model_filename)
joblib.dump(X.columns.tolist(), columns_filename)

print(f"\nModel saved to '{model_filename}'")
print(f"Feature columns saved to '{columns_filename}'")


Model saved to 'random_forest_model.joblib'
Feature columns saved to 'model_features.joblib'


Model Usage (example)

In [17]:
def get_automated_data():
    """
    This function simulates fetching data from a weather or agricultural API.
    In a real-world scenario, this would make an API call based on the
    user's location, crop type, and time of year.
    For this example, we'll return fixed values.
    """
    return {
        'Annual_Rainfall': 1200.0,
        'Fertilizer': 500.0,
        'Pesticide': 150.0,
        'Crop_Year': 2020,
    }

In [18]:
def predict_yield(manual_inputs):
    """
    Predicts crop yield for new input data.
    Takes user-provided manual inputs and combines them with automated data.
    """
    try:
        loaded_model = joblib.load(model_filename)
        loaded_columns = joblib.load(columns_filename)

        # Get automated data from the simulated API call
        automated_data = get_automated_data()

        # Combine manual and automated data
        input_data = {**manual_inputs, **automated_data}

        input_df = pd.DataFrame([input_data])
        for col in loaded_columns:
            if col not in input_df.columns:
                input_df[col] = 0

        input_df = input_df[loaded_columns]

        prediction = loaded_model.predict(input_df)

        return prediction[0]
    except Exception as e:
        return f"An error occurred during prediction: {e}"

# Example of how to use the prediction function with user inputs
# A frontend would gather this data from a web form.
user_inputs = {
    'Area': 150.0,
    # The one-hot encoded columns need to be matched.
    'Crop_Rice': 1,
    'State_Uttar Pradesh': 1,
    'Season_Kharif': 1
}

In [19]:
print("\n--- Example Prediction with User Inputs ---")
predicted_yield = predict_yield(user_inputs)
print(f"Predicted Yield for the example input: {predicted_yield:.2f}")


--- Example Prediction with User Inputs ---
Predicted Yield for the example input: 2.56


Yield Maximization Part